# Name That LLM �� G07 Technical Report

18-class source attribution �� prompt-grouped development �� two-stage LoRA �� offline T4 inference

**Best verified Public BA: 0.674**. This is a CPU-only, self-contained report and aggregate-results check; it does not train models, load private inputs, generate predictions or submit to the competition.

[GitHub report](https://github.com/fjnuslw/name-that-llm) �� [Competition](https://www.kaggle.com/competitions/name-that-llm)

Development OOF, Public scores and engineering samples have different evaluation scopes. No Private score, final rank, medal or independent unbiased evaluation is claimed.


# �������о�ȡ��

## ����������

���������ı������ 18 ��ָ����Դģ��֮һ��Balanced Accuracy Ϊ 18 �� Recall �ľ�ֵ���ٷ� 2,699 ������ 150 �� Prompt���̶����۸� 30 Prompt��ͬһ����ı������ۡ�ÿ���ٷ������� OOF ֻ����δ���� Prompt �������ģ�͡�

�����ⲿ��Ϊ Open Pool Kaggle V16 / open-pool-v16-r1��223,840 ����61 ��������Դ����Ŀ�˲��ظ������ǩͬ�ġ��ٷ�ԭ�ľ�ȷ�غ����Ѷ�λ����������������Դ�Ͱ汾���ݡ��������� V16 �����ı�������δ����ɾ������֪��������в���������ģ�����ݻ�ȫ��������ⶼ����֤���ĸ��ٷ�����ƥ��/����ֶν�������Դ��������ѵ�����������Ȩ�ء�

ѡ��ʹ�ð��༰��Դ�ֲ㡢�޷ŻصĹ̶� 54k��ÿ�� 3,000������ʽѵ��ʹ�������ⲿ�ء�54k �Ǽ���Ԥ��ѡ��22.384 ��Ҳδ������Ŀ�ܿ�����ʵ����֤Ϊ����������

## ���������ɱ��������

���Ǹ�Ϊ Qwen3.8-27B��64 block��5,120 ά���� Qwen3.5-9B-Base��32 block��4,096 ά����ԭ���ҽض��� 1,024 token���������� token ������ģ�塣�� final RMSNorm ǰ�� block ����� masked mean����������� L2��ƴ�Ӳ����� sqrt(K)���� 18 ������ͷ��

�ȱȽ�ȫ�����㡢ĩ�㡢final norm�����Ȱ˲㡢Top-8 ��̰�ķ��������������㡢ɾ���뻻�㡣003 ������ѡԤ��Ϊ 27B 768��9B 384��������ѡ������������Ӹ��ˡ�

| �Ǹ� | ����ѡ��Ĳ㼯�ϣ�1 ��ʼ�� | ������ BA ��ֵ / SD |
|---|---|---|
| 27B | 19,22,33,34,47,48,49,50,51,52,53,54 | 0.480939 / 0.003586 |
| 9B | 6,14,17,18,19,20,21,22,23,24,32 | 0.458456 / 0.006375 |

��ѵ���������Ӿ�ֵΪ 0.421662/0.395481����������ͬһ���� OOF��û�б���δ���� Lockbox����Щ��Ԥ���ں�ѡ������ȫ�����Ż� LoRA ������֤����

## ���׶� LoRA �� matched frozen ����

���۹��ó�ʼ�����С��߼� batch��ͷ�Ż��䷽��������ÿ���������ⲿ���� 1 epoch���ٴ��ⲿ���������������ٷ�ѵ���۸� 2 epochs��������ȫ�ٷ� final��BF16 ѵ����LoRA r=16��alpha=32��dropout=0������ѡ�� block ʵ�ʴ��ڵ� nn.Linear���߼� batch 16��

�ⲿ/�ٷ��׶� LoRA ѧϰ��Ϊ 1e-4/2e-5��ͷѧϰ�� 5e-4/2e-4��AdamW��weight decay 0.01��3% warmup ���ݶȲü� 1�����ݶ����ݶ�·��ѡ������ģ�飺27B ���� 54 �㣬���� 57�C64 ����������޷�Ӱ�쵱ǰ������ʧ����ʵѵ���������ݶȺͲ������¡�

005 Ϊ 9B ������� centered Ridge ����ʳ߶ȡ�ÿ����ѵ�� Prompt �ڷ� 80%/20% �����/У׼������ȫ������ѵ��������� centered ͷ��������֤��ǩ����϶�Ӧͷ��У׼���ڲ�У׼���ѱ���Ӧ LoRA/plain ͷ�������ڲ��ɼ����Ƕ�����֤��

## �������������з�֧

P0 ���� Omer V1 �����������ε� Transformer ����������plain/twin/Gaussian��TF-IDF/SVC�����ͳ�ơ�����ջ������ ATLAS���������ڵ��������á����в���Ϊ���ݰ汾������ѡ��ʵ�顢27B/9B ѵ�����ա���������ͷ/У׼��ϡ���֧�滻������������Ч����֤��

���� 27B final �� 9B ���۷ֱ������滻ԭ����β����֧������Ծ�����Ԥ���ظ����ӡ��� P0 �� 27B �������ʻ�ϣ���ÿ�ۼ��� 9B У׼ log-prob��ִ��ԭ��ͬ������ƽ�������幫ʽ���ֿ� README ���ں�ʾ����


In [ ]:
import json, math, hashlib, csv
from pathlib import Path

METRICS = json.loads('{"format": "name-that-llm-public-report-v1", "snapshot_date": "2026-11-16", "task": {"classes": 18, "official_rows": 2699, "prompt_groups": 150, "cv_folds": 5}, "data": {"release": "open-pool-v16-r1", "kaggle_version": 16, "external_rows": 223840, "credited_sources": 61, "selection_rows": 54000, "selection_per_class": 3000}, "layer_selection": {"qwen38-27b": {"layers": [19, 22, 33, 34, 47, 48, 49, 50, 51, 52, 53, 54], "development_ba_mean": 0.48093876874637503, "seed_sd": 0.0035855538951600797, "seeds": [{"seed": 42, "ba": 0.4860054685558042}, {"seed": 20261112, "ba": 0.4785806611981109}, {"seed": 20261113, "ba": 0.47823017648521}]}, "qwen35-9b-base": {"layers": [6, 14, 17, 18, 19, 20, 21, 22, 23, 24, 32], "development_ba_mean": 0.4584563758389262, "seed_sd": 0.006375094139605285, "seeds": [{"seed": 42, "ba": 0.4674695500869997}, {"seed": 20261112, "ba": 0.45376336067611234}, {"seed": 20261113, "ba": 0.4541362167536664}]}}, "matched_development_oof": [{"model": "qwen38-27b", "arm": "frozen", "ba": 0.4726820780512056}, {"model": "qwen38-27b", "arm": "lora", "ba": 0.6509321401938851}, {"model": "qwen35-9b-base", "arm": "frozen", "ba": 0.4426696495152871}, {"model": "qwen35-9b-base", "arm": "lora", "ba": 0.6309072831220482}], "readout_development_oof": {"plain": 0.6305369127516779, "centered": 0.6078846631866767, "calibrated": 0.6712652249565001}, "public_scores": [{"experiment": "V3", "a27": 0.12, "b9": 0.4, "public_ba": 0.669, "submission_ref": 58339923}, {"experiment": "G01", "a27": 0.25, "b9": 0.4, "public_ba": 0.666, "submission_ref": 58363608}, {"experiment": "G04", "a27": 0.4, "b9": 0.4, "public_ba": 0.672, "submission_ref": 58363612}, {"experiment": "G07", "a27": 0.55, "b9": 0.4, "public_ba": 0.674, "submission_ref": 58364991}, {"experiment": "A27", "a27": null, "b9": null, "public_ba": 0.64, "submission_ref": 58365339}, {"experiment": "A09", "a27": null, "b9": null, "public_ba": 0.626, "submission_ref": 58365344}], "inference_180_rows": {"27b": {"rows": 180, "members": 1, "before_forward_seconds": 158.23564515600003, "after_forward_seconds": 103.80736635100004, "time_reduction": 0.34396977211639446}, "9b": {"rows": 180, "members": 5, "before_forward_seconds": 260.562636258, "after_forward_seconds": 181.25759484699995, "time_reduction": 0.30436075774300564}}, "008_status_checked_utc": "2026-11-16T09:51:30.886628+00:00", "008_status": [{"experiment": "N01", "status": "error", "submission_status": "SubmissionStatus.COMPLETE", "public_ba": null, "runtime_timeout": true}, {"experiment": "N02", "status": "submitted", "submission_status": "SubmissionStatus.PENDING", "public_ba": null, "runtime_timeout": false}, {"experiment": "N03", "status": "submitted", "submission_status": "SubmissionStatus.PENDING", "public_ba": null, "runtime_timeout": false}], "008_example_runtime": [{"run": "shared-v1", "rows": 3, "selected_rows": 1, "total_seconds": 12958.808849583, "expert_seconds": 10181.390401321001, "all_25_segments": true}, {"run": "N02-v2", "rows": 3, "selected_rows": 1, "total_seconds": 12176.431473701, "expert_seconds": 9648.870746494, "all_25_segments": true}, {"run": "N03-v1", "rows": 3, "selected_rows": 1, "total_seconds": 11805.980533963, "expert_seconds": 8819.447452305001, "all_25_segments": true}], "evaluation_boundary": {"oof_is_development": true, "layer_selection_reused_development": true, "inner_calibration_groups_seen_by_lora": true, "independent_lockbox_remains": false, "private_score_available": false, "hidden_runtime_available": false, "without_p0_dual_model_public_control_available": false}}')
EVIDENCE_MANIFEST = json.loads('{"format": "name-that-llm-evidence-manifest-v1", "metrics_sha256_canonical_json": "3648407ad220e9b5f7770c11e0a8deb4c4c808b77e8b0d039129ad4766f31d04", "archive_fingerprints": [{"stage": "003", "archive": "20261112T091407Z.zip", "sha256": "355938cab8c9eab18c3a1ded0551fb98b7d1493d248c3eddabedbf247fc2d4ca"}, {"stage": "004", "archive": "20261113T065343Z-parallel.zip", "sha256": "cdd70a46f0830c2e1e46ea9285ece750114b1eb5f10727d3af8fd270d64c53a8"}, {"stage": "005", "archive": "20261113T171219Z.zip", "sha256": "7a7c81cff6ad53451e7eba1ac1d566e423321f37ca52630e70f088b4a9183af0"}], "score_records": [{"experiment": "V3", "a27": 0.12, "b9": 0.4, "public_ba": 0.669, "submission_ref": 58339923}, {"experiment": "G01", "a27": 0.25, "b9": 0.4, "public_ba": 0.666, "submission_ref": 58363608}, {"experiment": "G04", "a27": 0.4, "b9": 0.4, "public_ba": 0.672, "submission_ref": 58363612}, {"experiment": "G07", "a27": 0.55, "b9": 0.4, "public_ba": 0.674, "submission_ref": 58364991}, {"experiment": "A27", "a27": null, "b9": null, "public_ba": 0.64, "submission_ref": 58365339}, {"experiment": "A09", "a27": null, "b9": null, "public_ba": 0.626, "submission_ref": 58365344}], "validation": "20 key OOF files recomputed locally; ID, label, Prompt and fixed-fold alignment checked", "shared_payload": "aggregates, reporting code and dependency references", "raw_text_ids_predictions_weights_or_credentials_included": false}')
print('Snapshot:', METRICS['snapshot_date'])
print('Task:', METRICS['task'])
print('Data:', METRICS['data'])


# ʵ�������۱߽�

������ֵ���Ա���ԭʼ OOF �ش���ٷ����ֻ�ִ�����������������ܣ�ԭʼ 20 �ݹؼ� OOF ���ڱ������¼��� argmax��18 �� Recall ��ֵ�����˶� ID/��ǩ/Prompt/�̶��ۺš��ش� ZIP ָ��λ�� results/evidence-manifest.json��

## ͬ���� LoRA ���գ����� OOF��

| �Ǹ� | Frozen | LoRA | BA ��ֵ |
|---|---:|---:|---:|
| 27B | 0.472682 | 0.650932 | +0.178250 |
| 9B | 0.442670 | 0.630907 | +0.188238 |

003 ѡ���� 004 ѵ��ʹ�õ���������ͷ�䷽��ͬ������ֱ��������� LoRA��004 �� matched �ȽϹ̶�����Щ����������ǰ�ܹ�ѡ��������ͬһ��������004 ˫ģ���ں����Ϊ 0.657604���� G07 ����ͬһ�����䷽��

005 9B plain/centered/calibrated ���� OOF Ϊ 0.630537/0.607885/0.671265���ڲ�У׼�鱻��Ӧ LoRA/plain ���������ܰѸ÷�����Ϊȫ������ƫǶ����֤��

## �ѵ÷� Public ����

| �汾 | ��һ�׶� P0/27B | �ڶ��׶� 9B log share | Public BA |
|---|---|---:|---:|
| V3 | 88/12 | 40% | 0.669 |
| G01 | 75/25 | 40% | 0.666 |
| G04 | 60/40 | 40% | 0.672 |
| G07 | 45/55 | 40% | **0.674** |
| A27 | ���� 27B ����֧ | �� | 0.640 |
| A09 | ���� 9B ���� | �� | 0.626 |

G07 �� V3 �� 0.005 BA���� 0.5 ���ٷֵ㡣����Ȩ�����߲���ȫ�̵�����δ�����ռ�ȣ��������ҵ����ŵ㡣����֧����У׼�빲ͬ������ȱ�١�27B+9B���� P0���Ķ��գ���˲��ܽ�����ϵͳ�Ե���֧��ȫ����ֵ���� P0�����������б�ǩ��������С���������� Private ��Ȼ��ߡ�

## ������Ϣ���ĺ���ʵ�飨δ��ɣ�

�̶��ʲ���������������ֱ������ P0 ������˫ģ�͡����� 9B Ȩ�ء��Ƴ� centered ��ϡ��������� P0 �͵������� GLM ר�ҡ�����ȱʧ���հ�δ�⴦�����������ܲ�ɼ���


In [ ]:
scores = {r['experiment']: r['public_ba'] for r in METRICS['public_scores']}
for r in METRICS['public_scores']:
    print(f"{r['experiment']:>3}  Public BA={r['public_ba']:.3f}  submission={r['submission_ref']}")
delta = scores['G07'] - scores['V3']
assert math.isclose(delta, .005, abs_tol=1e-12)
print(f"G07 vs V3: +{delta:.3f} BA = +{100*delta:.1f} percentage points")
print("No-P0 dual-model control: not measured")


In [ ]:
for model in ('qwen38-27b', 'qwen35-9b-base'):
    arms = {r['arm']: r['ba'] for r in METRICS['matched_development_oof'] if r['model']==model}
    print(model, 'matched development OOF:', arms, 'delta:', arms['lora']-arms['frozen'])
for branch, r in METRICS['inference_180_rows'].items():
    reduction = 1-r['after_forward_seconds']/r['before_forward_seconds']
    assert math.isclose(reduction, r['time_reduction'], abs_tol=1e-12)
    print(f"{branch}: same 180 rows, {r['members']} member(s), forward time reduction {100*reduction:.1f}%")
print("Scope: branch forward time only, not full-system/hidden runtime")


# ˫ T4 ������ʧ�ܸ���

## ����ع���

A800 ѵ������������Ǹɸ�������ʽ LoRA/ͷ�ݶ�ͬ���������߼� batch�������Ż�����RNG������λ��������������û���Ѻ�ʵ��ͨ�ö࿨���Լ��ٱ�����

˫ T4 ���������У�27B ���� NF4 Ȩ���� FP16 ���㣬9B ���� FP16������һ�� 9B �������л����� adapter���̶�ģ�� revision��tokenizer ���ú����˳��ʹ������ wheelhouse��˳������ݴ桢�ӽ����ͷ��Դ棬�Լ� ID/����/Parquet ��Լ��顣

ͬ 180 �С���Ӧδ�� Prompt �۵� 9B plain �����Է����� BF16��NF4 �� argmax һ���� 90.56%��FP16 99.44%��֧��ѡ�� FP16���ⲻ��ȫϵͳ BA ��ȫ���ı�����֤����27B Ҳû��ͬ final ������ BF16/NF4 �ȼ�֤����

| ��֧��ͬһ 180 �У� | ԭ���� | ������������ | ǰ���ʱ | ���� |
|---|---|---|---|---:|
| 27B final | NF4��batch 4 | ��� 16 �У�4096 padded token | 158.24��103.81 �� | 34.4% |
| 9B ���ۺϼ� | FP16��batch 8 | ��� 32 �У�4096 padded token | 260.56��181.26 �� | 30.4% |

�����ָ� ID ˳��plain argmax �ֱ� 180/180��900/900 һ�£�9B У׼�� 899/900���̶��ں� 180/180 һ�¡�����ֻ��Ӧ��֧ǰ��δ�������м��ء��������λ����ؼ������� GPU reserved ͳ���ڼ��غ����ã�ֻ����ǰ�򣬲��ܱ�Ϊȫ���Դ��ֵ��

V3 ����ȫ����Լ 43.47 ���ӣ�G07 Լ 43.29 ���ӣ���ʽ�÷�˵����Ӧ�汾��ɹ�ƽ̨�������ܣ������ؽ׶κ�ʱ���ɼ���

## 008����ʵ GLM ��Ȼר��

��ʵ��� P0 һ�� 14B �ӷ�֧��Ϊ Nuri v4/Nishant ���� adapter ʵ��ǰ���� 72:28 ������ϣ��������������ںϼ�ԭ����֮�����ʵ GLM5.3/Flash �� teacher-forced ��Ȼ֤�ݡ�

���ù����� 54 ά������ 18 �� LogisticRegression ״̬��δ��ѵ��ר�ҡ�������า�� 20% �ı���ÿ�� 160 token���� 130k ���� token-row Ԥ��Լ������С�������� 1 �У���������ʵ�ʴ���һ�С�ȫ�� 6 �� Flash��19 �� GLM ��Ƭ��ִ�С�

����ȫ����Ϊ 3.28�C3.60 Сʱ��ר�Ҳ���Ϊ 2.45�C2.83 Сʱ��״̬����ʱ�䣨UTC����2026-11-16T09:51:30.886628+00:00��

N01: hidden runtime timeout, score=None; N02: SubmissionStatus.PENDING, score=None; N03: SubmissionStatus.PENDING, score=None

С������Ч������������ģ 9 Сʱ��ͨ��������/��Ƭ I/O���̶��ɱ��� token ǰ����Ҫһ��Ԥ�㣻��־�в��ܾ�ȷ�ֽ����س�ʱ���򡣸÷�֧��Ϊʧ�������չʾ�����ܳ��ȶ������Ż���


## G07 outer-fusion interface

The next cell shares the standalone formula. It requires NumPy only when imported and external aligned probabilities/postprocessing. No learned assets or full public pipeline are included.

In [ ]:
FUSION_SOURCE = '"""G07 outer fusion. Public-parent/postprocessing implementations are external.\n\nFormula follows the credited Omer V1 pipeline. This standalone interface\nexposes the project\'s configurable outer shares; it does not ship trained assets.\n"""\nimport numpy as np\n\ndef softmax(x):\n    x = np.asarray(x, dtype=np.float64)\n    x = x - x.max(axis=-1, keepdims=True)\n    e = np.exp(x)\n    return e / e.sum(axis=-1, keepdims=True)\n\ndef fuse(parent, own27, own9_logp, postprocess, a27=.55, b9=.4):\n    """postprocess receives N x 18 probabilities and returns probabilities."""\n    parent = np.asarray(parent, dtype=np.float64)\n    own27 = np.asarray(own27, dtype=np.float64)\n    own9_logp = np.asarray(own9_logp, dtype=np.float64)\n    if not (0 <= a27 <= 1 and 0 <= b9 < 1):\n        raise ValueError("invalid shares")\n    if parent.ndim != 2 or parent.shape[1] != 18:\n        raise ValueError("parent must be N x 18")\n    if own27.shape != parent.shape or own9_logp.shape != (5, *parent.shape):\n        raise ValueError("unaligned branch shapes")\n    for p in (parent, own27, np.exp(own9_logp)):\n        if not np.isfinite(p).all() or (p < 0).any() or not np.allclose(p.sum(-1), 1, atol=1e-6):\n            raise ValueError("invalid probabilities")\n    p1 = (1-a27)*parent + a27*own27\n    output = []\n    for lp in own9_logp:\n        q = softmax((1-b9)*np.log(np.maximum(p1, 1e-15)) + b9*lp)\n        p = np.asarray(postprocess(q), dtype=np.float64)\n        if p.shape != parent.shape or not np.isfinite(p).all() or (p < 0).any() or not np.allclose(p.sum(-1), 1, atol=1e-6):\n            raise ValueError("postprocess must return aligned probabilities")\n        output.append(p)\n    return np.mean(output, axis=0)\n'
compile(FUSION_SOURCE, 'fusion.py', 'exec')
print(FUSION_SOURCE)


# ֤�ݡ������빫���߽�

�������� 2026-11-16 �ı���֤�ݿ���Ϊ������Public ������ִ�˶�����Ϊ 2026-11-15��OOF Ϊ������ϣ�Public Ϊƽ̨�������ܽ��������/180 ��Ϊ������֤�����Էֿ���¼��

## �Ѹ���

- �ٷ� 2,699 �С�150 Prompt��18 �ࣻ�̶� CV ���۸� 30 Prompt��
- 54,000 ��Ψһ�ⲿ ID��ÿ�� 3,000��
- 20 �ݹؼ� OOF ������ ID����ǩ��Prompt�����й̶��ۺš����� logits��argmax �� BA��
- �ѵ÷ְ汾 V3/G01/G04/G07/A27/A09 �ı��عٷ� API ��ִ��
- ͬ�����֧Ч���� 008 ʵ��ִ������¼��

����������ԭʼ�ı������� ID/Ԥ�⡢ѵ��Ȩ�ء�ƾ�ݻ��ڲ���ά�ļ���ָ�ƿ����ڻ�úϷ�ԭʼ�鵵������ݺ��飻�����������ű�ֻ��֤�ѹ������ܵ������������������������� OOF��ѵ����ƽ̨���֡�

## ��������Ҫ��

����ǰͨ���ٷ� Competition API ��ȡ���� rules��foundational-rules �� Code Requirements��������ش������˵�����Ȩ�������ڱ��������� Kaggle Notebook �������������в������ṩ������ͨ���԰����Ĺ��� Notebook ����ͬһ������������룬���� Apache-2.0�������·ַ��ٷ�ԭ�Ļ������Ȩ�ء�

[��������](https://www.kaggle.com/competitions/name-that-llm/rules) �� [����˵��](https://www.kaggle.com/competitions/name-that-llm)

## ��������Դ

���� P0��Gaussian/twin��TF-IDF/����֧��ATLAS �빲ͬ��������ԭʼ��������Ŀ���׷�Χ˵�����Զ��϶�Ϊ���˶������ȫ��������Ҳ����������Ԥ���ģ��Ϊԭ����������Դ�� THIRD-PARTY-NOTICES.md��

����һ����ժҪ SHA256��3648407ad220e9b5f7770c11e0a8deb4c4c808b77e8b0d039129ad4766f31d04


# Third-party sources and scope

This repository publishes newly written explanatory material, aggregate results
and lightweight reporting/fusion code. It does not redistribute third-party
training data, checkpoints, prefit heads, compiled binaries or full pipelines.
Their original licenses and attribution requirements continue to apply.

- Competition: https://www.kaggle.com/competitions/name-that-llm
- Open Pool Kaggle V16, crtproduction and credited source contributors:
  https://www.kaggle.com/datasets/crtproduction/name-that-llm-open-pool/versions/16
- Qwen3.8-27B, fixed revision 37561816e66b5b9153b1b64c10968c18c1e6f4a0:
  https://huggingface.co/Qwen/Qwen3.8-27B/tree/37561816e66b5b9153b1b64c10968c18c1e6f4a0
- Qwen3.5-9B-Base, revision 68c46c4b3498877f3ef123c856ecfde50c39f404:
  https://huggingface.co/Qwen/Qwen3.5-9B-Base/tree/68c46c4b3498877f3ef123c856ecfde50c39f404
- Omer Faruk Merey, fixed V1, scriptVersionId 367805055:
  https://www.kaggle.com/code/omerfarukmerey/0-682-0-681-qwen3-8-27b-gsq-12-mix-0-50?scriptVersionId=367805055
  Retain the original upstream credits, including lightningv08, heliosli,
  mercurydust, Roman Tamrazov and other contributors.
- Public P0 assets also credit Dhruv Tibarewal, Nishant Dahal, Nuri Dudak,
  Lucas Boesen and the authors of the original model, encoder, style and SVC assets.
  A versioned reference inventory is provided in results/public-dependencies.json.
- ATLAS assets, josephadamski91, V3:
  https://www.kaggle.com/datasets/josephadamski91/ntl-atlas-qwen3-8b-v1/versions/3
- 008 likelihood extension: Lightning V52 / feiyu V1 and their upstream
  feature/classifier/execution contributors:
  https://www.kaggle.com/code/lightningv08/0-694-glm-5-3-likelihood-qwen3-8-27b-lora
  https://www.kaggle.com/code/feiyu26/ntl-0684-v1-graft?scriptVersionId=369151207
  This experiment is not presented as a trained or reliably deployed original expert.

The standalone fusion formula follows the credited arithmetic/log-probability
outer pipeline. Its callback intentionally leaves original common postprocessing
external; this is not the full scoring implementation. The repository's
Apache-2.0 license applies only to newly authored material supplied here.


In [ ]:
VERIFY_SOURCE = '"""Check the published evidence digest and arithmetic, without private data."""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport math\n\nroot = Path(__file__).resolve().parents[1]\nm = json.loads((root / "results/metrics.json").read_text(encoding="utf-8"))\ne = json.loads((root / "results/evidence-manifest.json").read_text(encoding="utf-8"))\ncanonical = json.dumps(m, sort_keys=True, ensure_ascii=False, separators=(",", ":"))\nassert hashlib.sha256(canonical.encode()).hexdigest() == e["metrics_sha256_canonical_json"]\nscores = {r["experiment"]: r["public_ba"] for r in m["public_scores"]}\nassert math.isclose(scores["G07"]-scores["V3"], .005, abs_tol=1e-12)\nassert max(scores.values()) == scores["G07"]\nassert all(0 <= r["ba"] <= 1 for r in m["matched_development_oof"])\nassert m["data"]["selection_per_class"] * m["task"]["classes"] == m["data"]["selection_rows"]\nfor branch, r in m["inference_180_rows"].items():\n    assert math.isclose(1-r["after_forward_seconds"]/r["before_forward_seconds"], r["time_reduction"], abs_tol=1e-12)\nprint("Published evidence digest and arithmetic: passed")\nprint("G07 - V3: +0.005 BA = +0.5 percentage points")\nprint("Scope: aggregate consistency; does not rerun training, OOF or hidden scoring")\n'
compile(VERIFY_SOURCE, 'verify_results.py', 'exec')
print(VERIFY_SOURCE)


In [ ]:
CANONICAL = json.dumps(METRICS, sort_keys=True, ensure_ascii=False, separators=(',', ':'))
assert hashlib.sha256(CANONICAL.encode()).hexdigest() == '3648407ad220e9b5f7770c11e0a8deb4c4c808b77e8b0d039129ad4766f31d04'
out = Path.cwd()
for folder in ('results','src','scripts'): (out/folder).mkdir(exist_ok=True)
(out/'results/metrics.json').write_text(json.dumps(METRICS, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
(out/'results/evidence-manifest.json').write_text(json.dumps(EVIDENCE_MANIFEST, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
(out/'src/fusion.py').write_text(FUSION_SOURCE, encoding='utf-8')
(out/'scripts/verify_results.py').write_text(VERIFY_SOURCE, encoding='utf-8')
print('Aggregate evidence digest: passed')
print('Exported results/, src/fusion.py and scripts/verify_results.py')
print('No training, weights, original texts, raw row predictions or competition submission')
